# Phase 1 Notebook: Scaffold + Data Quality Baseline

## What was done
- Created the repository structure for ingestion, identity, features, forecasting, optimisation, agent, backtesting, evaluation, storage, and utils.
- Added typed contracts and baseline configs.
- Added tests and a sample fixture dataset for pipeline smoke checks.

## Why it was done
- To keep each later phase focused on implementation, we establish a runnable and testable platform now.
- We also establish an auditable quality baseline before building production ingestion and models.

In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path('../tests/fixtures/raw/fpl_players_sample.csv')
df = pd.read_csv(data_path)
df.head()

## Data quality checks
We check shape, missingness, duplicated player-gameweek keys, and temporal consistency.

In [ ]:
key_cols = ['player_id', 'season', 'gameweek']
missing = df.isna().sum().sort_values(ascending=False)
duplicate_keys = df.duplicated(subset=key_cols).sum()
availability_issues = (df['available_pre_deadline'] == 0).sum()

summary = {
    'rows': len(df),
    'columns': len(df.columns),
    'duplicate_key_rows': int(duplicate_keys),
    'rows_not_available_pre_deadline': int(availability_issues),
}
summary, missing[missing > 0]

## Findings and anomalies
- One sample row is marked unavailable before deadline (`available_pre_deadline = 0`).
- This row is intentional for testing leakage-safe filtering behavior.
- No duplicate player-season-gameweek records in the fixture.

## How anomalies are handled
- Rows unavailable before deadline are excluded in feature generation by `enforce_pre_deadline_only`.
- In later phases, stale-source anomalies will be handled with fail-soft ingestion and explicit freshness metadata.